In [2]:
from qdrant_client import QdrantClient

In [3]:
client = QdrantClient(url="http://localhost:6333")

In [4]:
client.get_collections()

CollectionsResponse(collections=[])

In [5]:
from openai import OpenAI

client_openai = OpenAI()

In [6]:
embedding = client_openai.embeddings.create(
    model="text-embedding-3-small",
    input="How do I join the course?"
)

In [7]:
len(embedding.data[0].embedding)

1536

In [8]:
from qdrant_client.models import Distance, VectorParams

In [9]:
client.create_collection(
    collection_name="course-questions",
    vectors_config=VectorParams(
        size=1536,
        distance=Distance.COSINE
    )
)

True

In [10]:
from rag import load_faq_documents

documents = load_faq_documents()
len(documents)

1400

In [11]:
texts = [
    f"Question: {doc['question']}\nAnswer: {doc['answer']}"
    for doc in documents
]

In [12]:
texts[0]

"Question: How do I access the course modules and materials?\nAnswer: All course materials are in the GitHub repo. Each module has its own folder (e.g. `01-overview`, `03-mcp`), and cohort-specific homework is under the `cohorts/` folder.\n\nFor the 2026 cohort, the week-by-week folders are:\n\n- Week 1: [01-ai-native-workflow](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/01-ai-native-workflow)\n- Week 2: [02-development](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/02-development)\n- Week 3: [03-deployment](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/03-deployment)\n- Week 4: [04-devops](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/04-devops)\n\nStart from the folder for your current week — it gathers everything for that week in one place.\n\nLectures are pre-recorded and available in the YouTube playlist. New workshops or updated videos are announced on

In [13]:
embeddings_response = client_openai.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

In [14]:
vectors = [
    item.embedding
    for item in embeddings_response.data
]

In [15]:
len(vectors), len(vectors[0])

(1400, 1536)

In [16]:
from qdrant_client.models import PointStruct

In [17]:
points = [
    PointStruct(
        id=i,
        vector=vectors[i],
        payload=documents[i]
    )
    for i in range(len(documents))
]

In [19]:
batch_size = 100

for i in range(0, len(points), batch_size):
    client.upsert(
        collection_name="course-questions",
        points=points[i:i + batch_size]
    )

In [20]:
client.count(
    collection_name="course-questions"
)

CountResult(count=1400)

In [21]:
query = "How do I join the course?"

In [22]:
query_embedding = client_openai.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

In [23]:
results = client.query_points(
    collection_name="course-questions",
    query=query_embedding,
    limit=5
)

In [24]:
results.points[0]

ScoredPoint(id=53, version=1, score=0.627772, payload={'id': '8fd8cd336d', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How do I sign up?', 'answer': 'In the course [GitHub repository](http://mlzoomcamp.com), there’s a link to sign up. Here it is: [courses.datatalks.club](https://courses.datatalks.club/register/ml-zoomcamp/)'}, vector=None, shard_key=None, order_value=None)

In [25]:
context = "\n\n".join(
    f"Question: {point.payload['question']}\nAnswer: {point.payload['answer']}"
    for point in results.points
)

In [26]:
prompt = f"""
Answer the QUESTION based on the CONTEXT.

QUESTION:
{query}

CONTEXT:
{context}
"""

In [27]:
response = client_openai.responses.create(
    model="gpt-4o-mini",
    input=prompt
)

In [28]:
response.output_text

"To join the course, you can sign up by visiting the course [GitHub repository](http://mlzoomcamp.com) where you'll find a link to register. Here it is: [courses.datatalks.club](https://courses.datatalks.club/register/ml-zoomcamp/)."